# 4장. 에이전트 설정과 재사용

같은 저장 Agent의 기본 지침과 Session별 변경을 비교합니다.

| 연결 기준 | 이번 장 |
| --- | --- |
| 공통 예제 | 동일한 sales.csv의 매출 분석 |
| 새로 배울 기능 | Agent 설정 재사용·세션별 override |
| 이번 실습 조합 | 저장 Agent 하나 · 독립 Session 두 개 · 메시지 입력 |


<!-- colab-link -->
[독립 실행본 다운로드 안내](../../docs/colab.md) · Colab 업로드용


## 학습 안내

| 구분 | 3장 | 이 장 |
| --- | --- | --- |
| 재사용 대상 | 같은 Session | 같은 저장 Agent |
| Session 수 | 하나 | 두 개 |
| 입력 방식 | 기존 Session에 후속 입력 | 매번 새 Session 생성 |
| 학습 목표 | 대화·파일을 이어 사용 | 기본 설정 재사용과 세션별 변경 |

같은 Agent ID를 사용해도 이전 Session의 대화나 파일을 가져오는 것은 아닙니다. 두 Session에 CSV를 각각 전달합니다. 두 분석 모두 같은 모델·입력·결과 형식을 사용하며, 비교 대상은 세션별 instructions입니다.

| 단계 | 직접 사용하는 API | 확인할 내용 |
| --- | --- | --- |
| 저장 Agent 생성 | `agents.create()` | Agent ID·모델·기본 지침 |
| 기본 실행 | `sessions.create(agent_id=...)` | 기본 Session·완료 상태·분석 JSON |
| override 실행 | 같은 Agent ID와 inline 지침 | 다른 Session·완료 상태·분석 JSON |
| 설정 비교 | `sessions.retrieve()`, `agents.retrieve()` | 모델 유지·지침 변경 범위·원본 유지 |
| 정리 | 정리 helper의 삭제·재조회 | Session 두 개와 Agent 한 개의 API상 부재 |

수동 실습은 아래 셀을 위에서 아래로 실행합니다. 자동 검증은 `uv run --group dev python scripts/run_intro.py --chapter 4 --live`로 실행합니다. 모델 교체의 품질·비용 비교나 지침 준수율 평가는 이 장의 실습에 포함하지 않습니다.


**실습 안내:** 각 장은 새 커널에서 독립 실행합니다. 실행 모드·응답과 파일의 차이·검증 기록은 [공통 학습·실습 안내](../../docs/validation.md)를 참고하세요.


## 0. 실습 준비와 실행 모드

로컬에서는 저장소 루트에서 `uv sync --locked`, `uv run jupyter lab`을 실행하고 `.venv` 커널을 선택하세요.
Colab에서는 다운로드 안내의 독립 실행본을 업로드하세요. 준비 셀이 패키지를 직접 설치하고 포함된 코드·데이터를 생성합니다.
실제 API 실습은 **RUN_LIVE=True**로 명시적으로 허용합니다.
키는 `.env.local` 또는 Colab Secrets에서 읽으며 출력하지 않습니다. 라이브 실행에는 비용이 발생합니다.


In [ ]:
import os
import sys

if "google.colab" in sys.modules and os.environ.get("TUTORIAL_OFFLINE_CHECK") != "1":
    raise RuntimeError("This is the local source notebook. Upload the standalone version from the download guide to Colab.")
print("Use the local .venv kernel. Install dependencies with: uv sync --locked")


In [ ]:
from tutorial_helpers.common import load_settings

RUN_LIVE = False
MODEL_OVERRIDE = ""
NUMBER = 4
CHAPTER = "04_agent_configuration"
client = None
settings = load_settings(RUN_LIVE, MODEL_OVERRIDE)
print(
    {
        "chapter": NUMBER,
        "live_enabled": settings.enabled,
        "model": settings.model or "Offline mode",
    }
)


## 1. 저장할 설정과 매번 보낼 입력 준비하기

모델·기본 지침은 저장 Agent에 보관하고, CSV와 이번 질문은 각 Session의 입력으로 전달합니다. 아래는 로컬 CSV 확인이며 API 호출이 아닙니다. 두 Session에서 같은 CSV를 새로 저장하고 분석하므로, 파일 공유를 전제로 하지 않습니다.


In [ ]:
csv_text = (settings.root / "data/sales.csv").read_text()
print(csv_text)


### 요청 구성

`BASE_INSTRUCTIONS`는 Python으로 계산하고 입력에 없는 거래를 만들지 않도록 정합니다. `METRICS_CONTRACT`는 최종 JSON 형식이며 실제 입력 메시지에 조합합니다. 환경은 OpenAI-hosted sandbox로, 표준 라이브러리만 사용하므로 외부 네트워크를 끕니다. 이 장은 텍스트 JSON과 적용 설정을 조회하며 파일을 다운로드하지 않습니다.


In [ ]:
BASE_INSTRUCTIONS = (
    "You are a careful sales analyst. Use Python "
    "for all calculations. Use only the provided "
    "CSV; never invent rows. Do not browse the web"
    " or delegate. Keep progress commentary short."
    " Return only the JSON object requested by the"
    " user as the final answer, with no Markdown "
    "fences."
)
METRICS_CONTRACT = (
    'Return exactly {"row_count": integer, '
    '"total_quantity": integer, "total_revenue": '
    'integer, "by_product": {product_name: '
    "integer_revenue}}. Revenue is quantity "
    "multiplied by unit_price. Do not round "
    "integer amounts."
)
task = (
    "Save the CSV below as /workspace/sales.csv. "
    "Write and run a Python standard-library analysis script, inspect stdout, "
    "and calculate the summary. "
    + METRICS_CONTRACT
    + "\nCSV input:\n"
    + csv_text
)


### 세션별 override는 무엇을 바꾸나요?

`agent_id`는 저장 Agent의 기본값을 사용하도록 지정합니다. 세션 생성에 `agent={"instructions": ...}`를 함께 전달하면 그 Session의 instructions를 지정한 값으로 바꿉니다. 생략한 모델 설정은 저장 Agent의 값을 사용합니다.

기본 지침을 유지하면서 설명 방식만 추가하려고 **Python에서 기본 문자열과 추가 문장을 결합**합니다. API가 두 지침을 자동으로 이어 붙이는 예제가 아닙니다. 아래 출력에서 추가 문장과 변경할 필드를 확인하세요. 응답이 짧아 보이는지보다 실제 적용 설정 조회를 통해 변경 범위를 확인합니다.


In [ ]:
EXTRA_INSTRUCTIONS = " For this session, use a compact presentation in commentary."
override_instructions = BASE_INSTRUCTIONS + EXTRA_INSTRUCTIONS
environment = {
    "type": "openai_hosted",
    "container_size": "small",
    "network": {"access": "disabled"},
}
print({"overridden_field": "instructions", "added_instruction": EXTRA_INSTRUCTIONS.strip()})
print("The model, input CSV, environment configuration, and JSON contract stay the same.")


## 2. 저장 Agent 생성하기

먼저 재사용할 모델·지침을 `agents.create()`로 저장합니다. Session 생성과는 별개인 자원이므로 Agent ID도 추적합니다. Agent 설정·이름·metadata에 API 키를 넣지 않습니다.

이후 네 실행 단계에서 `cleanup_on_error()`가 오류·일반적인 인터럽트 시 정리를 시도합니다. 정상 경로에서는 **6절 정리 셀까지 실행**하세요. 셀 사이에서 중단할 때도 정리 셀을 실행해야 합니다. 강제 커널 종료는 자동 정리를 보장하지 못합니다.


#### API 실행 허용 확인

설정 셀에서 `RUN_LIVE=True`를 선택한 뒤 이 준비 셀을 실행하세요. 기본 False에서는 여기서 오류 메시지와 함께 멈춥니다. 이후 API 셀에는 실행 여부 분기를 반복하지 않습니다. `requires_api` 태그는 오프라인 검증기용이며 Jupyter 자체의 접근 제어가 아닙니다.


In [ ]:
from openai import OpenAI

client = None
if not settings.enabled:
    raise RuntimeError(
        "Set RUN_LIVE=True to run the API example. Use the validation script for offline checks."
    )
client = OpenAI(api_key=settings.api_key, timeout=90.0, max_retries=0)


### 이번 실행 상태 준비

`output_dir`는 분석 보고서가 아니라 복구용 기록을 저장하는 경로입니다. 상태 준비는 새 실행 시작 시 한 번만 실행하세요. 진행 중 다시 실행하면 자원 ID와 제출 기록을 잃을 수 있습니다.

Agent 생성 응답을 받으면 ID를 즉시 추적하고 `save_agent_reference()`로 `agent.json`에 run ID와 Agent ID만 기록합니다. 이 helper는 API 호출·실행을 감싸지 않습니다. 세션 생성 시도 metadata는 별도의 `creation.json`에 기록합니다.

각 분석 타이머는 제출 직전에 시작합니다. 600초 제한은 이벤트 도착 시 확인하며, 무응답 중 강제 중단하는 전체 타이머가 아닙니다. 클라이언트 HTTP timeout은 90초입니다.


In [ ]:
from tutorial_helpers.common import (
    new_session_metadata, save_agent_reference,
    cleanup_resources, cleanup_on_error, check_event, safe_error,
)
from uuid import uuid4
import json
import time


In [ ]:
run_id = uuid4().hex
output_dir = settings.root / "outputs/intro" / CHAPTER / run_id
output_dir.mkdir(parents=True, exist_ok=True)
sessions, agents, attempts = {}, {}, []
cleanup_errors = []
saved = base_result = custom_result = base_turn = custom_turn = None
agent_creation_attempted = base_attempted = override_attempted = False


### 저장 Agent의 기준 설정 확인

다음 셀은 저장 Agent를 한 번 생성합니다. 출력된 Agent ID·모델·기본 지침을 확인하세요. 아직 분석 Session을 생성하거나 작업을 실행한 단계는 아닙니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if agent_creation_attempted:
        raise RuntimeError("Agent creation was already attempted. Inspect saved state first.")
    agent_creation_attempted = True
    saved = client.beta.agents.create(
        model=settings.model,
        instructions=BASE_INSTRUCTIONS,
        name=f"tutorial-sales-{run_id[:8]}",
        metadata={"tutorial_run_id": run_id},
    )
    agents[saved.id] = {"deleted": False}
    save_agent_reference(output_dir, run_id=run_id, agent_id=saved.id)
    print({"agent_id": saved.id, "model": saved.model, "instructions": saved.instructions})
    print("Agent recovery reference:", output_dir / "agent.json")


## 3. 기본 설정으로 새 Session 실행하기

`agent_id=saved.id`를 전달하고 inline Agent 설정은 생략합니다. 매번 보낼 입력과 환경은 Session 생성 요청에 직접 지정합니다. 환경 설정이 같아도 두 요청이 같은 대화·파일 공간을 사용하는 것은 아닙니다.

02장의 스트리밍 흐름으로 텍스트를 표시하고 최종 응답을 수집합니다. 예상하지 않은 외부 작업은 오류로 처리하며 `check_event()`로 실패·취소를 감지합니다. 이벤트 수나 순서를 고정하지 않고, `idle`만으로 성공을 판단하지 않습니다.

첫 결과와 조회한 root Turn의 완료 상태를 확인한 뒤 기본 분석 JSON을 표시합니다. JSON 파싱 성공은 필드 형식이나 계산 정확성을 보장하지 않습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if saved is None:
        raise RuntimeError("Create the saved Agent first.")
    if base_attempted:
        raise RuntimeError("This session creation was already attempted. Do not resubmit.")
    base_attempted = True
    started = time.monotonic()
    with client.beta.agents.sessions.create(
        agent_id=saved.id,
        environment=environment,
        input=task,
        metadata=new_session_metadata(
            CHAPTER, run_id, "base", attempts=attempts, output_dir=output_dir,
        ),
        stream=True,
    ).with_result_collection() as stream:
        for event in stream:
            if event.type == "agent.session.created":
                sessions[event.session.id] = {
                    "deleted": False,
                    "environment_id": getattr(event.session.environment, "id", None),
                }
                print("Session:", event.session.id)
            elif event.type == "agent.session.turn.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "agent.session.requires_action":
                raise RuntimeError("Unexpected external action in this hosted exercise")
            else:
                check_event(event)
            if time.monotonic() - started > 600:
                raise TimeoutError("Event-observed execution deadline exceeded")
        base_result = stream.get_final_result()
        sessions.setdefault(base_result.session_id, {"deleted": False})
    base_turn = client.beta.agents.sessions.turns.retrieve(
        base_result.turn_id, session_id=base_result.session_id, timeout=15,
    )
    if base_result.turn.status != "completed" or base_turn.status != "completed":
        raise RuntimeError("Root turn completion was not confirmed")
    print("\nBase analysis:")
    print({"session_id": base_result.session_id, "turn_id": base_result.turn_id,
           "status": base_turn.status})
    print(json.dumps(json.loads(base_result.output_text), indent=2))


## 4. 같은 Agent로 지침만 바꾼 Session 실행하기

이번에도 `sessions.create()`로 **새 Session**을 만듭니다. 03장의 후속 입력과 달리 `sessions.stream(existing_session_id, ...)`을 사용하지 않습니다. 같은 `agent_id`와 `agent={"instructions": override_instructions}`를 함께 전달합니다.

입력 CSV와 환경 설정을 다시 전달하며 모델은 override하지 않습니다. 변경된 실행의 Session ID·완료 상태·분석 JSON을 확인하세요. 분석값이 같다고 override 적용을 증명하는 것은 아니므로 다음 절에서 설정을 직접 조회합니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if saved is None:
        raise RuntimeError("Create the saved Agent first.")
    if override_attempted:
        raise RuntimeError("This session creation was already attempted. Do not resubmit.")
    override_attempted = True
    started = time.monotonic()
    with client.beta.agents.sessions.create(
        agent_id=saved.id,
        agent={"instructions": override_instructions},
        environment=environment,
        input=task,
        metadata=new_session_metadata(
            CHAPTER, run_id, "override", attempts=attempts, output_dir=output_dir,
        ),
        stream=True,
    ).with_result_collection() as stream:
        for event in stream:
            if event.type == "agent.session.created":
                sessions[event.session.id] = {
                    "deleted": False,
                    "environment_id": getattr(event.session.environment, "id", None),
                }
                print("Session:", event.session.id)
            elif event.type == "agent.session.turn.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "agent.session.requires_action":
                raise RuntimeError("Unexpected external action in this hosted exercise")
            else:
                check_event(event)
            if time.monotonic() - started > 600:
                raise TimeoutError("Event-observed execution deadline exceeded")
        custom_result = stream.get_final_result()
        sessions.setdefault(custom_result.session_id, {"deleted": False})
    custom_turn = client.beta.agents.sessions.turns.retrieve(
        custom_result.turn_id, session_id=custom_result.session_id, timeout=15,
    )
    if custom_result.turn.status != "completed" or custom_turn.status != "completed":
        raise RuntimeError("Root turn completion was not confirmed")
    print("\nOverride analysis:")
    print({"session_id": custom_result.session_id, "turn_id": custom_result.turn_id,
           "status": custom_turn.status})
    print(json.dumps(json.loads(custom_result.output_text), indent=2))


## 5. 실제 적용 설정과 저장 Agent 비교하기

두 Session과 원래 Agent를 API로 조회합니다. 같은 Session을 재사용했는지가 아니라 **저장한 기본값이 각 Session에 어떻게 적용됐는지** 확인합니다.

| 확인 항목 | 기대 결과 |
| --- | --- |
| 두 Session ID | 서로 다름 |
| 두 Session의 모델 | 처음 지정한 모델과 같음 |
| 기본 Session의 지침 | 처음 지정한 기본 지침과 같음 |
| override Session의 지침 | override 지침과 같음 |
| 재조회한 저장 Agent | 처음 지정한 모델·기본 지침 유지 |

설정 조회·Turn 완료·계산 정확성은 서로 다른 확인입니다. 아래 비교는 설정 적용 범위를 확인하고, Python 명령·계산값 정답 검사는 별도 검증기가 수행합니다. 응답 문체의 실제 준수율은 평가하지 않습니다.

이 장은 저장 Agent의 update를 호출하지 않습니다. 세션 생성 시 지침을 지정하는 것과 저장 Agent 자체를 수정하는 것은 별개입니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if base_turn is None or custom_turn is None:
        raise RuntimeError("Complete both analysis cells before comparing settings.")
    base_session = client.beta.agents.sessions.retrieve(base_result.session_id, timeout=15)
    custom_session = client.beta.agents.sessions.retrieve(custom_result.session_id, timeout=15)
    saved_again = client.beta.agents.retrieve(saved.id, timeout=15)
    configuration_checks = {
        "distinct_sessions": base_session.id != custom_session.id,
        "base_model_matches": base_session.agent.model == settings.model,
        "override_model_matches": custom_session.agent.model == settings.model,
        "base_instructions_match": base_session.agent.instructions == BASE_INSTRUCTIONS,
        "override_instructions_match": custom_session.agent.instructions == override_instructions,
        "saved_model_unchanged": saved_again.model == settings.model,
        "saved_instructions_unchanged": saved_again.instructions == BASE_INSTRUCTIONS,
    }
    print(json.dumps({"agent_id": saved_again.id,
                      "base_session_id": base_session.id,
                      "override_session_id": custom_session.id,
                      "checks": configuration_checks}, indent=2))
    if not all(configuration_checks.values()):
        raise RuntimeError("Applied settings differ from the intended configuration")


## 6. Session 두 개와 저장 Agent 정리하기

결과와 설정을 확인했으므로 이번 실행에서 만든 자원을 정리합니다. 중간에 학습을 중단하더라도 이 셀을 실행하세요. 스트림이 끝나도 Session이나 저장 Agent가 자동으로 삭제되는 것은 아닙니다.

`cleanup_resources()`는 추적한 Session들의 취소·삭제·부재 확인을 시도한 다음 저장 Agent를 삭제·재조회합니다. 한 자원의 정리가 실패해도 다른 자원 정리를 시도합니다. 정리 오류는 성공으로 표시하지 않고, 기존 실행·검증 오류가 있으면 그 원인을 보존합니다.

정상 전체 실행의 출력에는 Session 두 개와 Agent 한 개가 있으며 각각 `deleted`·`verified_absent`를 확인합니다. 이는 API상 부재 확인이며 물리적 sandbox 정리의 동기 완료를 뜻하지 않습니다. 부분 실행에서는 실제로 생성한 자원만 나타납니다.

삭제 후 로컬 결과·설정 객체와 복구 기록은 남습니다. 삭제한 자원에 대한 추가 서버 조회는 사용할 수 없습니다.


In [ ]:
try:
    print("Cleaning up this run's sessions and saved Agent...", flush=True)
finally:
    original_error = sys.exc_info()[1]
    try:
        cleanup_errors[:] = cleanup_resources(client, sessions, agents, attempts)
    except Exception as cleanup_error:
        cleanup_errors.append(safe_error(cleanup_error))
    if cleanup_errors:
        if original_error is None:
            raise RuntimeError("Resource cleanup incomplete; inspect local records")
        original_error.add_note("Resource cleanup incomplete; inspect local records")
if not all(owned.get("verified_absent") for owned in [*sessions.values(), *agents.values()]):
    raise RuntimeError("Resource absence was not confirmed")
print({"session_cleanup": sessions, "agent_cleanup": agents})


## 7. 실패 시 조치와 다음 장

- 401/403은 인증·프로젝트 접근 권한, 429는 API 제한을 확인합니다. 키나 원본 오류 body는 출력하지 않습니다.
- 연결 오류 뒤에는 정리 결과와 저장된 ID를 먼저 확인합니다. 같은 생성 셀을 무조건 재실행하면 새 자원을 만들 수 있습니다. 이 실습은 생성 시도 플래그로 같은 실행 상태에서 중복 제출을 막습니다.
- 강제 종료 시 `outputs/intro/<chapter>/<run_id>/agent.json`의 Agent ID와 `creation.json`의 Session 생성 metadata로 이번 실행만 식별합니다. 로컬 파일이 남아 있다는 사실만으로 자원이 아직 존재한다고 판단하지 않습니다. 검증 도구로 실행했다면 `summary.json`도 참고하세요.
- Agent 생성 응답 자체를 받지 못했다면 `agent.json`과 추적 ID가 없을 수 있습니다. 현재 helper는 이 Agent를 자동 검색·회수하지 않습니다. 프로젝트 Agent 목록에서 run ID metadata·이름을 확인해 소유 자원을 식별하고, 다른 실습·운영 Agent를 삭제하지 마세요. 일반적인 복구 절차는 [빠른 시작](../quickstart.ipynb)을 참고하세요.

완료 시 두 분석 JSON과 설정 비교 결과, Session 두 개·Agent 한 개의 삭제 확인을 읽을 수 있어야 합니다. [5장](05_files_artifacts.ipynb)에서는 같은 CSV를 파일로 공급하고 분석 JSON·Markdown 보고서를 다운로드합니다.


## 공식 참고 자료

### Agents API 전용 문서

- [Configuring Agents](https://developers.openai.com/api/docs/guides/agents-api/configuration) — Agent 저장·재사용과 세션별 설정 재정의.

링크 확인: 2026-11-03. 문서 링크 확인은 API 실습 실행 검증과 별개입니다.
